In [56]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

Project root: /home/vinchar/llmops-demo
Data dir: /home/vinchar/llmops-demo/training_data
Adapter output dir: /home/vinchar/llmops-demo/adapters
Base model: Qwen/Qwen2.5-7B-Instruct
Adapters: ('finance', 'legal', 'healthcare')


## Download adapters from MLFlow

In [46]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(settings_cfg.mlflow_tracking_uri)
mlflow.set_experiment(settings_cfg.mlflow_experiment_name)
client = MlflowClient()
experiment = client.get_experiment_by_name(settings_cfg.mlflow_experiment_name)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else "not created yet")

Tracking URI: http://mlflow.mlflow.svc.cluster.local:5000
Experiment: llmops-demo


In [61]:
from pathlib import Path
import shutil
from mlflow.artifacts import download_artifacts

adapter_root = Path("/home/vinchar/llmops-demo/adapters")
adapter_root.mkdir(parents=True, exist_ok=True)

downloaded_adapters = {}

for adapter in settings_cfg.adapters:

    adapter_runs = runs[
        (runs["tags.adapter_name"] == adapter)
        & (runs["status"] == "FINISHED")
    ].sort_values("start_time", ascending=False)

    if adapter_runs.empty:
        print(f"No successful run found for {adapter}")
        continue

    run_id = adapter_runs.iloc[0]["run_id"]

    print(f"Downloading '{adapter}' from run {run_id}")

    # Download artifacts
    downloaded_path = download_artifacts(
        artifact_uri=f"runs:/{run_id}/adapter",
        dst_path=str(adapter_root),
    )

    nested = final_dir / "adapter"

    if nested.exists():
        for item in nested.iterdir():
            shutil.move(str(item), str(final_dir))
    
        nested.rmdir()

    downloaded_path = Path(downloaded_path)

    # Final desired location
    final_dir = adapter_root / adapter

    # Remove old adapter dir if it exists
    if final_dir.exists():
        shutil.rmtree(final_dir)

    # Rename/move downloaded adapter dir
    shutil.move(str(downloaded_path), str(final_dir))

    downloaded_adapters[adapter] = final_dir

    print(f"Downloaded {adapter} to {final_dir}")

print("\nFinal adapter locations:")
for name, path in downloaded_adapters.items():
    print(f"{name}: {path}")

Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10
Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10
Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10
Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10
Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10


Downloaded finance to /home/vinchar/llmops-demo/adapters/finance


Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10
Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10
Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10


Downloaded legal to /home/vinchar/llmops-demo/adapters/legal


Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10
Connection pool is full, discarding connection: local-s3-service.ezdata-system.svc.cluster.local. Connection pool size: 10


Downloaded healthcare to /home/vinchar/llmops-demo/adapters/healthcare

Final adapter locations:
finance: /home/vinchar/llmops-demo/adapters/finance
legal: /home/vinchar/llmops-demo/adapters/legal
healthcare: /home/vinchar/llmops-demo/adapters/healthcare


## Copy adapters to vLLM Model

In [64]:
!/home/vinchar/kube cp \
  /home/vinchar/llmops-demo/adapters \
  project-public/qwen257b-predictor-00001-deployment-59df6df457-gwj2k:/tmp/adapters \
  -c kserve-container

In [67]:
!/home/vinchar/kube exec -it \
  -n project-public \
  qwen257b-predictor-00001-deployment-59df6df457-gwj2k \
  -c kserve-container -- \
  bash -c "ls -l /tmp/adapters/"

total 12
drwxr-xr-x 3 root root 4096 Jun  4 12:16 finance
drwxr-xr-x 2 root root 4096 Jun  4 12:16 healthcare
drwxr-xr-x 2 root root 4096 Jun  4 12:16 legal


## Load adapters

Expected output: one success line per adapter.

In [75]:
for adapter in settings_cfg.adapters:

    try:
        print(f"Loading adapter: {adapter}")

        load_adapter(
            settings_cfg.vllm_base_url,
            settings_cfg.vllm_api_key,
            adapter,
            Path(f"/tmp/adapters/{adapter}"),
        )

    except RuntimeError as e:

        if "already been loaded" in str(e):
            print(f"{adapter} already loaded, skipping.")
        else:
            raise

print("Done.")

Loading adapter: finance
Loaded adapter 'finance' from /tmp/adapters/finance
Loading adapter: legal
Loaded adapter 'legal' from /tmp/adapters/legal
Loading adapter: healthcare
healthcare already loaded, skipping.
Done.


## Verify vLLM model registration

Expected output: `finance`, `legal`, and `healthcare` appear in `/v1/models`.

In [79]:
import requests

response = requests.get(
    f"{settings_cfg.vllm_base_url}/v1/models",
    headers={"Authorization": f"Bearer {settings_cfg.vllm_api_key}"},
    timeout=10,
)
response.raise_for_status()
print(response.json())

{'object': 'list', 'data': [{'id': 'Qwen/Qwen2.5-7B-Instruct', 'object': 'model', 'created': 1780576443, 'owned_by': 'vllm', 'root': 'Qwen/Qwen2.5-7B-Instruct', 'parent': None, 'max_model_len': 32768, 'permission': [{'id': 'modelperm-bb1dd1aeb416ba2c', 'object': 'model_permission', 'created': 1780576443, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}, {'id': 'healthcare', 'object': 'model', 'created': 1780576443, 'owned_by': 'vllm', 'root': '/tmp/adapters/healthcare', 'parent': 'Qwen/Qwen2.5-7B-Instruct', 'max_model_len': None, 'permission': [{'id': 'modelperm-8ea4de7641afe8b4', 'object': 'model_permission', 'created': 1780576443, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None